# Decision Tree Random Forest #

### Imports ###



In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier



### Load Cleaned Dataset ###

In [33]:
script_dir = os.getcwd()
rel_path = "intermediate"

dataset = pd.read_csv(os.path.join(script_dir, rel_path, "CLEAN_data_injection_moulding.csv"))

### Feature Selection ###

Feature selection need not be too agressive. It's important just to check that we're feeding only process/ sensor variables. We will remove features that are almost always constant as well as irrelevant information.

In [34]:
features = [
    'ZSx [s]',        # injection time
    'ACPx [cm³]',     # cushion volume
    'ZDx [s]',        # dosing time
    'ZUs [s]',        # cycle time
    'ZEx [s]',        # (keep if it’s meaningful in your process)
    'GEx [kWh]',      # energy consumption
    'H16x [°C]',      # temperature 1
    'H10x [°C]'       # temperature 2
]

X = dataset[features].copy()
y = dataset['ASZ [Sch]'].astype(int).copy()

print("X shape:", X.shape)
print("y positives:", y.sum(), " / ", len(y))


X shape: (4854, 8)
y positives: 163  /  4854


### Splitting and Scaling Data ###

In [35]:
def split_and_scale(X, y, test_size=0.2, random_state=42):
    # Stratify keeps defect ratio similar in train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )
    
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)
    
    return X_train, X_test, y_train, y_test, X_train_sc, X_test_sc, scaler

X_train, X_test, y_train, y_test, X_train_sc, X_test_sc, scaler = split_and_scale(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Train positives:", y_train.sum(), "Test positives:", y_test.sum())

Train size: (3883, 8) Test size: (971, 8)
Train positives: 130 Test positives: 33


### Baseline Model
This essentially does no machine learning and acts as a lower bound for comparison. It always just predicts the most frequent class. 

In [36]:
baseline = DummyClassifier(strategy="most_frequent", random_state=42)
baseline.fit(X_train, y_train)

y_pred_base = baseline.predict(X_test)

print("=== BASELINE ===")
print("F1:", f1_score(y_test, y_pred_base))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_base))
print(classification_report(y_test, y_pred_base, digits=3))

=== BASELINE ===
F1: 0.0
Confusion matrix:
 [[938   0]
 [ 33   0]]
              precision    recall  f1-score   support

           0      0.966     1.000     0.983       938
           1      0.000     0.000     0.000        33

    accuracy                          0.966       971
   macro avg      0.483     0.500     0.491       971
weighted avg      0.933     0.966     0.949       971



c:\Users\Anura\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Anura\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Anura\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### Decision Tree
Our decision trees might struggle with class imbalance because it may prioritize reducing overall error rather than focusing on our minority defect classes. Class weights solve this by assigning higher importance to minority classes during training. That's why we should select our class weight as balanced because defects are rare in our dataset. I also used mild regularization (max depth + min leaf size) to reduce overfitting

In [37]:
dt = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced",
    max_depth=6,
    min_samples_leaf=10
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("=== DECISION TREE ===")
print("F1:", f1_score(y_test, y_pred_dt))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, digits=3))


=== DECISION TREE ===
F1: 0.8354430379746836
Confusion matrix:
 [[925  13]
 [  0  33]]
              precision    recall  f1-score   support

           0      1.000     0.986     0.993       938
           1      0.717     1.000     0.835        33

    accuracy                          0.987       971
   macro avg      0.859     0.993     0.914       971
weighted avg      0.990     0.987     0.988       971



### Decision Tree Feature Importance
This shows which features the tree relied on most. We can check if that makes sense physically (challenge introducted hinted that importance should align with process understanding)

In [38]:
dt_importance = pd.Series(dt.feature_importances_, index=features).sort_values(ascending=False)
dt_importance


ZUs [s]       0.978900
ZEx [s]       0.008342
H10x [°C]     0.004795
H16x [°C]     0.003989
ACPx [cm³]    0.003226
GEx [kWh]     0.000733
ZDx [s]       0.000010
ZSx [s]       0.000007
dtype: float64

### Random Forest

As we know, a random forest reduces variance by averaging. We also see the forest’s feature importance ranking and can compare it to the decision tree to see which fits our process understanding.

In [39]:
rf = RandomForestClassifier(
    random_state=42,
    n_estimators=500,
    class_weight="balanced_subsample",
    min_samples_leaf=5,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== RANDOM FOREST ===")
print("F1:", f1_score(y_test, y_pred_rf))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, digits=3))

rf_importance = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
rf_importance


=== RANDOM FOREST ===
F1: 0.9428571428571428
Confusion matrix:
 [[934   4]
 [  0  33]]
              precision    recall  f1-score   support

           0      1.000     0.996     0.998       938
           1      0.892     1.000     0.943        33

    accuracy                          0.996       971
   macro avg      0.946     0.998     0.970       971
weighted avg      0.996     0.996     0.996       971



ZUs [s]       0.531936
H16x [°C]     0.287399
GEx [kWh]     0.081101
ZSx [s]       0.027525
ZDx [s]       0.025112
H10x [°C]     0.019472
ACPx [cm³]    0.014057
ZEx [s]       0.013399
dtype: float64

### Compare Models 

Now we compare the models by using the F1 score

In [40]:
results = pd.DataFrame({
    "Model": ["Baseline", "Decision Tree", "Random Forest"],
    "F1": [
        f1_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf)
    ]
}).sort_values("F1", ascending=False)

results


,Model,F1
2,Random Forest,0.942857
1,Decision Tree,0.835443
0,Baseline,0.000000
